# Composite indicator evidence, novel biomarker evidence, and references extracted from article documentation Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load dataset metadata from the Croissant schema,
- inspect available record sets and fields using their `@id` values,
- load tabular data into pandas DataFrames,
- perform exploratory data analysis with safe handling for mixed text/numeric content,
- create simple visualizations.

> Important: all dataset entities in this notebook are referenced by their Croissant `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn pyarrow

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the Croissant URL in a variable so the notebook is easy to reuse or adapt.

In [ ]:
import json
import re
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.0ff9-37as/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print(f'Name: {metadata.name}')
print(f'Description: {metadata.description}')
print(f'URL: {croissant_url}')

In [ ]:
# Display a compact metadata summary without treating metadata as a dictionary
summary = {
    'name': getattr(metadata, 'name', None),
    'description': getattr(metadata, 'description', None),
    'license': getattr(metadata, 'license', None),
    'cite_as': getattr(metadata, 'cite_as', None),
    'url': croissant_url,
}
pprint(summary)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Some Croissant datasets explicitly define record sets in metadata. Others may rely on distribution files and extraction rules. The code below first inspects metadata-defined record sets, then attempts to discover usable record sets dynamically.

In [ ]:
def safe_attr(obj, name, default=None):
    return getattr(obj, name, default)

metadata_record_sets = safe_attr(metadata, 'record_sets', None)
if metadata_record_sets is None:
    metadata_record_sets = safe_attr(metadata, 'recordSet', None)

record_set_rows = []
if metadata_record_sets:
    for rs in metadata_record_sets:
        rs_id = safe_attr(rs, 'id', None) or safe_attr(rs, '@id', None)
        rs_name = safe_attr(rs, 'name', None)
        rs_desc = safe_attr(rs, 'description', None)
        fields = safe_attr(rs, 'fields', None) or safe_attr(rs, 'field', None) or []
        record_set_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'description': rs_desc,
            'n_fields': len(fields),
        })

record_sets_df = pd.DataFrame(record_set_rows)
print('Metadata-defined record sets:')
display(record_sets_df if not record_sets_df.empty else pd.DataFrame({'message': ['No record sets explicitly listed in metadata.']}))

In [ ]:
# Try to inspect fields for each metadata-defined record set
field_rows = []

if metadata_record_sets:
    for rs in metadata_record_sets:
        rs_id = safe_attr(rs, 'id', None) or safe_attr(rs, '@id', None)
        rs_name = safe_attr(rs, 'name', None)
        fields = safe_attr(rs, 'fields', None) or safe_attr(rs, 'field', None) or []
        for field in fields:
            field_id = safe_attr(field, 'id', None) or safe_attr(field, '@id', None)
            field_name = safe_attr(field, 'name', None)
            field_desc = safe_attr(field, 'description', None)
            field_dtype = safe_attr(field, 'data_types', None) or safe_attr(field, 'dataType', None)
            source = safe_attr(field, 'source', None)
            source_id = safe_attr(source, 'id', None) if source is not None else None
            field_rows.append({
                'record_set_id': rs_id,
                'record_set_name': rs_name,
                'field_id': field_id,
                'field_name': field_name,
                'description': field_desc,
                'data_type': str(field_dtype),
                'source_id': source_id,
            })

fields_df = pd.DataFrame(field_rows)
display(fields_df.head(50) if not fields_df.empty else pd.DataFrame({'message': ['No field definitions available from metadata inspection.']}))
else:
    display(pd.DataFrame({'message': ['No record sets available to inspect fields from metadata.']}))

### Record set discovery
Because the provided metadata summary indicates `recordSet: []`, we use a discovery step to probe likely record set `@id` values. This is helpful when the underlying Croissant schema resolves record sets through distributions or extraction rules not shown in the compact summary.

In [ ]:
candidate_record_set_ids = [
    'composite_indicator_evidence',
    'novel_biomarker_evidence',
    'references',
    'composite-indicator-evidence',
    'novel-biomarker-evidence',
    'reference',
    'composite_indicator',
    'novel_biomarker',
]

discovered_record_sets = []

for rs_id in candidate_record_set_ids:
    try:
        sample = []
        for i, record in enumerate(dataset.records(record_set=rs_id)):
            sample.append(record)
            if i >= 1:
                break
        if sample:
            discovered_record_sets.append({'record_set_id': rs_id, 'sample_size_checked': len(sample), 'sample_keys': list(sample[0].keys())})
    except Exception:
        pass

discovered_record_sets_df = pd.DataFrame(discovered_record_sets)
display(discovered_record_sets_df if not discovered_record_sets_df.empty else pd.DataFrame({'message': ['No candidate record set IDs were automatically discovered.']}))

If any record set IDs are discovered, we can print example records directly. The code below uses record set `@id` values only.

In [ ]:
usable_record_sets = discovered_record_sets_df['record_set_id'].tolist() if not discovered_record_sets_df.empty else []

for rs_id in usable_record_sets:
    print(f'\nSample records from record set @id: {rs_id}')
    try:
        for i, x in enumerate(dataset.records(record_set=rs_id)):
            print(x)
            if i >= 1:
                break
    except Exception as e:
        print(f'Could not load records for {rs_id}: {e}')

## 3. Data Extraction
Load data from each discovered record set into a DataFrame for analysis.

This section is intentionally dynamic so the notebook works even when exact field layouts vary across record sets.

In [ ]:
dataframes = {}

for record_set_id in usable_record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set '{record_set_id}' with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Failed to load record set '{record_set_id}': {e}")

print(f'Loaded {len(dataframes)} DataFrame(s).')

In [ ]:
# Show columns and a preview for each loaded DataFrame
for record_set_id, df in dataframes.items():
    print(f'\nRecord set @id: {record_set_id}')
    print('Columns:')
    print(df.columns.tolist())
    display(df.head())

### Helper functions for mixed-format tabular evidence
The dataset description suggests that many fields may contain free-text evidence summaries, effect-size text, limitations, or references. The utilities below help identify numeric-looking columns or extract numeric values embedded in text.

In [ ]:
def extract_first_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    match = re.search(r'[-+]?\d*\.?\d+', str(value))
    return float(match.group()) if match else np.nan

def find_numeric_or_numeric_like_columns(df, min_non_null=3):
    candidates = []
    for col in df.columns:
        series = df[col]
        numeric_direct = pd.to_numeric(series, errors='coerce')
        direct_count = numeric_direct.notna().sum()
        extracted = series.map(extract_first_number)
        extracted_count = extracted.notna().sum()
        if direct_count >= min_non_null or extracted_count >= min_non_null:
            candidates.append({
                'field_id': col,
                'direct_numeric_values': int(direct_count),
                'numeric_like_values': int(extracted_count),
            })
    return pd.DataFrame(candidates).sort_values(['numeric_like_values', 'direct_numeric_values'], ascending=False) if candidates else pd.DataFrame(columns=['field_id', 'direct_numeric_values', 'numeric_like_values'])

def find_categorical_columns(df, max_unique=30):
    candidates = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        if 1 < nunique <= max_unique:
            candidates.append({'field_id': col, 'unique_values': int(nunique)})
    return pd.DataFrame(candidates).sort_values('unique_values') if candidates else pd.DataFrame(columns=['field_id', 'unique_values'])

In [ ]:
# Summarize candidate analysis fields by record set
for record_set_id, df in dataframes.items():
    print(f'\nCandidate numeric or numeric-like fields for record set @id: {record_set_id}')
    display(find_numeric_or_numeric_like_columns(df))
    print(f'Candidate categorical fields for record set @id: {record_set_id}')
    display(find_categorical_columns(df))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, text-to-number conversion, and grouping.

Because this dataset may be mostly literature-derived text with embedded numeric values, the EDA is designed to work robustly even if fields are not fully typed as numeric.

In [ ]:
# Choose a working record set dynamically
record_set_id = next(iter(dataframes.keys()), None)

if record_set_id is None:
    print('No DataFrames are available for EDA. Please inspect record set discovery results above.')
else:
    df = dataframes[record_set_id].copy()
    print(f'Using record set @id for EDA: {record_set_id}')
    print(f'Shape: {df.shape}')
    display(df.head())

In [ ]:
if record_set_id is not None:
    numeric_candidates = find_numeric_or_numeric_like_columns(df)
    categorical_candidates = find_categorical_columns(df)

    numeric_field = numeric_candidates.iloc[0]['field_id'] if not numeric_candidates.empty else None
    group_field = categorical_candidates.iloc[0]['field_id'] if not categorical_candidates.empty else None

    print(f'Selected numeric field @id: {numeric_field}')
    print(f'Selected group field @id: {group_field}')

In [ ]:
if record_set_id is not None and numeric_field is not None:
    df = df.copy()
    numeric_series = pd.to_numeric(df[numeric_field], errors='coerce')
    if numeric_series.notna().sum() < 3:
        numeric_series = df[numeric_field].map(extract_first_number)

    numeric_value_field = f'{numeric_field}__numeric'
    df[numeric_value_field] = numeric_series

    threshold = df[numeric_value_field].median(skipna=True)
    filtered_df = df[df[numeric_value_field] > threshold].copy()

    print(f'Filtered records with {numeric_field} > {threshold}:')
    display(filtered_df.head())

    std_val = filtered_df[numeric_value_field].std()
    normalized_field = f'{numeric_value_field}_normalized'
    if pd.notna(std_val) and std_val != 0:
        filtered_df[normalized_field] = (filtered_df[numeric_value_field] - filtered_df[numeric_value_field].mean()) / std_val
    else:
        filtered_df[normalized_field] = 0.0

    print(f'Normalized {numeric_field} for filtered records:')
    display(filtered_df[[numeric_field, numeric_value_field, normalized_field]].head())

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = (
            filtered_df[[group_field, numeric_value_field]]
            .dropna(subset=[group_field, numeric_value_field])
            .groupby(group_field, as_index=False)
            .mean(numeric_only=True)
            .sort_values(numeric_value_field, ascending=False)
        )
        print(f'Grouped data by {group_field}:')
        display(grouped_df.head())
    else:
        grouped_df = pd.DataFrame()
        print('No suitable grouping field was found for grouped analysis.')
else:
    filtered_df = pd.DataFrame()
    grouped_df = pd.DataFrame()
    print('Skipping numeric EDA because no suitable numeric field was identified.')

In [ ]:
# Missing-value overview for the selected record set
if record_set_id is not None:
    missing_summary = pd.DataFrame({
        'field_id': df.columns,
        'missing_count': [df[col].isna().sum() for col in df.columns],
        'missing_fraction': [df[col].isna().mean() for col in df.columns],
    }).sort_values('missing_fraction', ascending=False)
    display(missing_summary)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Plots are generated only when a usable numeric field is available.

In [ ]:
if record_set_id is not None and numeric_field is not None and numeric_value_field in df.columns:
    plot_df = df[[numeric_value_field]].dropna().copy()
    
    if not plot_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        sns.histplot(plot_df[numeric_value_field], kde=True, ax=axes[0], color='steelblue')
        axes[0].set_title(f'Distribution of {numeric_field}')
        axes[0].set_xlabel(numeric_field)
        
        sns.boxplot(x=plot_df[numeric_value_field], ax=axes[1], color='orange')
        axes[1].set_title(f'Boxplot of {numeric_field}')
        axes[1].set_xlabel(numeric_field)
        
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric values available to plot.')
else:
    print('Skipping numeric distribution plots because no suitable numeric field was available.')

In [ ]:
if not grouped_df.empty and group_field is not None and numeric_field is not None:
    top_grouped_df = grouped_df.head(10)
    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_grouped_df, x=group_field, y=numeric_value_field, palette='viridis')
    plt.title(f'Average {numeric_field} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping grouped bar plot because no grouped summary was created.')

## 6. Conclusion
This notebook demonstrated how to:
- load a dataset from a Croissant schema URL with `mlcroissant`,
- inspect dataset metadata safely through the metadata object,
- discover and load record sets using their `@id` values,
- convert record sets into pandas DataFrames,
- perform flexible EDA even when many fields contain text or mixed content,
- visualize numeric distributions and grouped summaries when numeric-like values are present.

For this dataset, the metadata suggests a literature-derived structure containing evidence summaries for composite indicators, novel biomarkers, and references. Depending on how the Croissant schema resolves record sets, you may need to refine the record set `@id` candidates above after inspecting the full schema.

Recommended next steps:
- inspect full schema details if a specific record set is not discovered automatically,
- map field `@id` values to domain concepts such as indicators, effect sizes, advantages, limitations, and references,
- build joins between evidence record sets and the references record set,
- add domain-specific text analysis for evidence synthesis.